# NB4 — Embeddings denses et embeddings de phrases avec tuning intégré

Ce notebook couvre `P16` à `P20`. La phase de tuning accéléré est particulièrement utile ici, car les embeddings denses peuvent coûter cher à recalculer ; on les calcule donc une seule fois par pipeline dans la phase d’optimisation.

Ce notebook conserve la **phase baseline sans optimisation**, puis ajoute une **phase d’optimisation accélérée**.
Le principe retenu est le suivant : pour chaque pipeline, on ajuste d’abord le **préprocesseur / vectoriseur une seule fois** sur un sous-ensemble d’apprentissage, puis on teste plusieurs réglages du **classifieur uniquement** sur les mêmes données déjà transformées. Cela réduit fortement le temps d’exécution.

Cette stratégie est très pratique pour explorer rapidement des réglages d’algorithmes, mais il faut bien comprendre qu’elle constitue une **optimisation accélérée**, plus pragmatique qu’une recherche entièrement relancée sur tout le pipeline à chaque itération.


In [ ]:
# Pour un run sur Colab

'''
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = "/content/drive/MyDrive/Disaster-Tweets-NLP"
MODELS_DIR = f"{PROJECT_ROOT}/notebooks/models_training"

%cd "{MODELS_DIR}"

import sys
if MODELS_DIR not in sys.path:
    sys.path.append(MODELS_DIR)

print("Projet :", PROJECT_ROOT)
print("Dossier courant :", MODELS_DIR)
'''


In [ ]:
# Installation éventuelle (décommente si nécessaire)
!pip install pandas numpy scikit-learn scipy matplotlib gensim sentence-transformers openpyxl mlflow

from collections import OrderedDict
from pathlib import Path
from contextlib import nullcontext
import json

import numpy as np
import pandas as pd
import mlflow

from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.preprocessing import Normalizer

from nlp_disaster_utils import (
    seed_everything,
    load_train_test_xy,
    round_results,
    metric_matrix_from_results,
    save_results_bundle,
    GensimMeanEmbeddingVectorizer,
    SentenceTransformerVectorizer,
    stratified_validation_split,
)
from mlflow_utils import (
    setup_mlflow_tracking,
    fit_evaluate_and_log_sklearn_pipeline,
)
from pipeline_tuning_utils import (
    split_pipeline_preprocessor_estimator,
    fit_transform_preprocessor_once,
    tune_classifier_on_fixed_features,
    evaluate_refit_outputs,
    compare_baseline_vs_tuned,
    log_tuning_run_to_mlflow,
    safe_scores,
)

seed_everything(42)


In [ ]:
DATA_DIR = ""
TRAIN_PATH = f"train.csv"
TEST_PATH = f"test.csv"

TEXT_COL = "text"
LABEL_COL = "target"
USE_AUX_TEXT_COLUMNS = False
LOWERCASE_TEXT = False
RANDOM_STATE = 42

OUTPUT_STEM = "NB4_classical_sentence_embeddings"
RESULTS_DIR = "outputs/NB4"
TUNING_OUTPUT_DIR = Path(RESULTS_DIR) / "tuning"
TUNING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Configuration MLflow
MLFLOW_EXPERIMENT_NAME = "DT_NB4_classical_sentence_embeddings"
MLFLOW_TRACKING_URI = Path("outputs/mlruns").resolve().as_uri()
MLFLOW_LOG_MODEL = False
USE_MLFLOW = True

tracking_uri = setup_mlflow_tracking(
    experiment_name=MLFLOW_EXPERIMENT_NAME,
    tracking_uri=MLFLOW_TRACKING_URI,
)
print("MLflow tracking URI :", tracking_uri)
print("MLflow experiment   :", MLFLOW_EXPERIMENT_NAME)

# Paramètres de tuning accéléré
VAL_SIZE_FOR_TUNING = 0.15
PRIMARY_TUNING_METRIC = "f1_pos"


MLflow tracking URI : file:///content/outputs/mlruns
MLflow experiment   : DT_NB4_classical_sentence_embeddings


/usr/local/lib/python3.12/dist-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


Même structure de configuration que les notebooks précédents. L'expérience MLflow est
enregistrée sous `DT_NB4_classical_sentence_embeddings`. À noter que contrairement aux
notebooks précédents, les chemins ici sont relatifs à la racine du dossier courant — si tu
tournes sur Colab, pense à adapter `DATA_DIR` et `RESULTS_DIR` vers les bons chemins Drive
(voir la cellule montée en commentaire tout en haut). Le tuning utilisera 15% du train comme
set de validation, avec le F1 de la classe 1 comme métrique de sélection.

In [ ]:
df_train, X_train, y_train, df_test, X_test, y_test = load_train_test_xy(
    train_path=TRAIN_PATH,
    test_path=TEST_PATH,
    text_col=TEXT_COL,
    label_col=LABEL_COL,
    use_extra_cols=USE_AUX_TEXT_COLUMNS,
    lowercase=LOWERCASE_TEXT,
)

print("Taille train :", len(X_train))
print("Taille test  :", len(X_test))
print("\nDistribution des classes - train :")
print(y_train.value_counts(normalize=True).sort_index())
print("\nDistribution des classes - test :")
print(y_test.value_counts(normalize=True).sort_index())


Taille train : 9096
Taille test  : 2274

Distribution des classes - train :
target
0    0.814094
1    0.185906
Name: proportion, dtype: float64

Distribution des classes - test :
target
0    0.813984
1    0.186016
Name: proportion, dtype: float64


On retrouve les mêmes proportions que dans les notebooks précédents : environ 9 096 tweets
en train et 2 274 en test, avec un déséquilibre 81/19 entre la classe 0 (Non-Disaster) et
la classe 1 (Disaster). Rien de nouveau ici — mais c'est toujours utile de le vérifier avant
de lancer des modèles aussi coûteux que les embeddings, pour s'assurer que les données
chargées sont bien celles attendues.

In [ ]:
pipelines = OrderedDict({
    "P16_GloVeTwitterMean_LogReg": Pipeline([
        ("embed", GensimMeanEmbeddingVectorizer(model_name="glove-twitter-200", normalize=True)),
        ("clf", LogisticRegression(max_iter=2500, C=1.0)),
    ]),
    "P17_GloVeTwitterMean_LinearSVC": Pipeline([
        ("embed", GensimMeanEmbeddingVectorizer(model_name="glove-twitter-200", normalize=True)),
        ("clf", LinearSVC(C=1.0)),
    ]),
    "P18_FastTextMean_LogReg": Pipeline([
        ("embed", GensimMeanEmbeddingVectorizer(model_name="fasttext-wiki-news-subwords-300", normalize=True)),
        ("clf", LogisticRegression(max_iter=2500, C=1.0)),
    ]),
    "P19_SentenceTransformer_LogReg": Pipeline([
        ("embed", SentenceTransformerVectorizer(model_name="sentence-transformers/all-MiniLM-L6-v2", batch_size=64)),
        ("clf", LogisticRegression(max_iter=2500, C=1.0)),
    ]),
    "P20_SentenceTransformer_LinearSVC": Pipeline([
        ("embed", SentenceTransformerVectorizer(model_name="sentence-transformers/all-MiniLM-L6-v2", batch_size=64)),
        ("clf", LinearSVC(C=1.0)),
    ]),
})


On teste ici 5 pipelines basés sur des représentations denses du texte, ce qui est un
changement important par rapport aux notebooks précédents qui travaillaient avec des vecteurs
creux (sparse) :

- **P16 & P17** : GloVe Twitter 200d + LogReg / LinearSVC — des embeddings pré-entraînés sur
  Twitter, donc potentiellement bien adaptés au registre informel et aux abréviations qu'on
  retrouve dans notre corpus. Chaque tweet est représenté par la moyenne des vecteurs de
  ses tokens.
- **P18** : FastText 300d (Wiki News) + LogReg — embeddings sous-lexicaux, plus robustes aux
  fautes d'orthographe et aux mots rares grâce à la décomposition en n-grammes de caractères.
  Le corpus d'entraînement (Wikipedia) est plus formel que Twitter, ce qui peut jouer dans
  les deux sens.
- **P19 & P20** : SentenceTransformer (all-MiniLM-L6-v2) + LogReg / LinearSVC — c'est le
  modèle le plus puissant de cette série. Il encode chaque tweet en un vecteur de 384
  dimensions qui capture le sens global de la phrase, pas juste la moyenne des mots. C'est
  ce qu'on attend de mieux sur cette tâche.

Le `normalize=True` dans GloVe et FastText met les vecteurs à l'échelle unitaire, ce qui
aide les classifieurs linéaires. Le `batch_size=64` dans SentenceTransformer accélère
l'inférence en traitant les tweets par lots.

## Phase 1 — Baselines sans optimisation

Cette première phase reproduit le benchmark initial : chaque pipeline est exécuté tel quel, avec ses paramètres de départ.

In [ ]:
resultats = []
baseline_failures = []

for nom_pipeline, pipeline in pipelines.items():
    print(f"Entraînement baseline -> {nom_pipeline}")
    display(pipeline)
    print("-" * 80)

    try:
        metrics = fit_evaluate_and_log_sklearn_pipeline(
            name=nom_pipeline,
            estimator=pipeline,
            X_train=X_train,
            X_test=X_test,
            y_train=y_train,
            y_test=y_test,
            notebook_name="CLASSICAL",
            family_name="classical_sentence_embeddings",
            output_dir=RESULTS_DIR,
            log_model=MLFLOW_LOG_MODEL,
        )
        resultats.append(metrics)
    except Exception as exc:
        baseline_failures.append({"pipeline": nom_pipeline, "error": str(exc)})
        print(f"Échec baseline pour {nom_pipeline} : {exc}")

baseline_df = round_results(pd.DataFrame(resultats))
display(baseline_df)

if baseline_failures:
    print("\nPipelines baseline en échec :")
    display(pd.DataFrame(baseline_failures))


Entraînement baseline -> P16_GloVeTwitterMean_LogReg


Pipeline(steps=[('embed', GensimMeanEmbeddingVectorizer(normalize=True)),
                ('clf', LogisticRegression(max_iter=2500))])

--------------------------------------------------------------------------------
[==================================================] 100.0% 758.5/758.5MB downloaded
Entraînement baseline -> P17_GloVeTwitterMean_LinearSVC


Pipeline(steps=[('embed', GensimMeanEmbeddingVectorizer(normalize=True)),
                ('clf', LinearSVC())])

--------------------------------------------------------------------------------
Entraînement baseline -> P18_FastTextMean_LogReg


...

Batches:   0%|          | 0/143 [00:00<?, ?it/s]

Batches:   0%|          | 0/36 [00:00<?, ?it/s]

Batches:   0%|          | 0/143 [00:00<?, ?it/s]

Batches:   0%|          | 0/36 [00:00<?, ?it/s]

pipeline,P16_GloVeTwitterMean_LogReg,P17_GloVeTwitterMean_LinearSVC,P18_FastTextMean_LogReg,P19_SentenceTransformer_LogReg,P20_SentenceTransformer_LinearSVC
train_accuracy,0.8764,0.8890,0.8677,0.8908,0.8967
train_precision_macro,0.8474,0.8504,0.8352,0.8482,0.8489
train_recall_macro,0.7069,0.7536,0.6829,0.7639,0.7887
train_f1_macro,0.7480,0.7887,0.7227,0.7959,0.8137
train_precision_weighted,0.8702,0.8827,0.8602,0.8846,0.8914
train_recall_weighted,0.8764,0.8890,0.8677,0.8908,0.8967
train_f1_weighted,0.8610,0.8801,0.8487,0.8833,0.8918
train_precision_class_0,0.8837,0.9018,0.8750,0.9061,0.9165
train_recall_class_0,0.9768,0.9691,0.9772,0.9660,0.9606
train_f1_class_0,0.9279,0.9343,0.9233,0.9351,0.9380


Les 5 pipelines ont tourné sans erreur. GloVe (758 MB) et FastText (958 MB) se sont
téléchargés automatiquement via Gensim au premier run — c'est normal, prévoir quelques
minutes. Pour SentenceTransformer, le warning sur `HF_TOKEN` est sans conséquence : le
modèle `all-MiniLM-L6-v2` (90 MB) s'est téléchargé sans authentification depuis HuggingFace.
Le message `embeddings.position_ids | UNEXPECTED` est également ignorable — c'est une clé
de poids présente dans le checkpoint source mais absente dans l'architecture cible, courant
quand on charge un modèle BERT pour une tâche différente de celle d'origine.

Sur le **set de test**, en se concentrant sur la classe 1 (Disaster) :
- **P19 (SentenceTransformer + LogReg)** et **P20 (SentenceTransformer + LinearSVC)**
  dominent avec les meilleurs F1 classe 1 (0.617 et 0.625) et les meilleures balanced
  accuracy (0.743 et 0.755). P20 détecte légèrement plus de vrais tweets disaster
  (recall 0.565 vs 0.530).
- **P17 (GloVe + LinearSVC)** se défend avec un F1 classe 1 de 0.585, au-dessus de
  **P16 (GloVe + LogReg)** à 0.530.
- **P18 (FastText + LogReg)** est le moins convaincant avec un F1 classe 1 de 0.491 —
  le corpus Wikipedia sur lequel FastText a été entraîné est visiblement moins adapté
  au registre Twitter que GloVe.

À noter : les écarts train/test sont raisonnables sur tous les pipelines, sans overfitting
franc. Les ROC-AUC entre 0.89 et 0.91 sont solides, mais le PR-AUC autour de 0.69-0.71
rappelle que la classe 1 reste difficile à capturer.

⚠️ Le tuning devra travailler principalement sur `class_weight='balanced'` pour améliorer
le recall des pipelines GloVe et FastText, qui restent trop prudents en baseline.

In [ ]:
help(fit_evaluate_and_log_sklearn_pipeline)

Help on function fit_evaluate_and_log_sklearn_pipeline in module mlflow_utils:

fit_evaluate_and_log_sklearn_pipeline(pipeline, X_train, y_train, X_valid, y_valid, run_name='baseline_run', model_name='model', tags=None, log_model=True)
    Fit + evaluate + log dans MLflow.

    Parameters
    ----------
    pipeline : sklearn Pipeline
    X_train, y_train : train set
    X_valid, y_valid : validation set
    run_name : str
    model_name : str
    tags : dict or None
    log_model : bool

    Returns
    -------
    dict
        métriques calculées



In [ ]:
save_results_bundle(pd.DataFrame(resultats), output_dir=RESULTS_DIR, stem=OUTPUT_STEM)
print(f"Fichiers CSV/XLSX baseline enregistrés dans {RESULTS_DIR}")


Fichiers CSV/XLSX baseline enregistrés dans outputs/NB4


Les résultats baseline sont sauvegardés en CSV et XLSX dans `outputs/NB4` avant de passer
au tuning. C'est important de le faire ici, avant la phase 2 : si le tuning plante sur un
pipeline lourd, on a quand même les résultats bruts de référence.

## Phase 2 — Tuning accéléré avec vectorisation unique par pipeline

Ici, pour chaque pipeline, on sépare le **préprocesseur** du **classifieur**. On ajuste le préprocesseur **une seule fois** sur un sous-ensemble d’apprentissage, puis on teste différentes combinaisons d’hyperparamètres du classifieur sur les mêmes données déjà vectorisées. Enfin, on réajuste le meilleur classifieur sur tout le train transformé une seule fois et on l’évalue sur train et test.

In [ ]:
X_fit, X_val, y_fit, y_val = stratified_validation_split(
    X_train,
    y_train,
    val_size=VAL_SIZE_FOR_TUNING,
    random_state=RANDOM_STATE,
)

print("Taille tuning-fit :", len(X_fit))
print("Taille tuning-val :", len(X_val))


Taille tuning-fit : 7731
Taille tuning-val : 1365


Création d'un sous-ensemble de validation stratifié (15 % du train) pour la phase de tuning :
**7 731 tweets** pour ajuster le vectoriseur et entraîner les classifieurs, et **1 365 tweets**
mis de côté pour comparer les combinaisons d'hyperparamètres. La stratification garantit que
la distribution des classes est préservée dans les deux splits.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Montage du Google Drive si les fichiers de données ou les modèles pré-entraînés y sont
stockés. Si les chemins configurés en début de notebook pointent déjà vers des fichiers
locaux à la session Colab, cette cellule peut être passée.

In [ ]:
classifier_param_grids = {
    "P16_GloVeTwitterMean_LogReg": {
        "C": [0.25, 0.5, 1.0, 2.0],
        "class_weight": [None, "balanced"],
    },
    "P17_GloVeTwitterMean_LinearSVC": {
        "C": [0.25, 0.5, 1.0, 2.0],
        "class_weight": [None, "balanced"],
    },
    "P18_FastTextMean_LogReg": {
        "C": [0.25, 0.5, 1.0, 2.0],
        "class_weight": [None, "balanced"],
    },
    "P19_SentenceTransformer_LogReg": {
        "C": [0.25, 0.5, 1.0, 2.0],
        "class_weight": [None, "balanced"],
    },
    "P20_SentenceTransformer_LinearSVC": {
        "C": [0.25, 0.5, 1.0, 2.0],
        "class_weight": [None, "balanced"],
    },
}


Grilles d'hyperparamètres à tester pour chaque classifieur. La grille est identique pour
les 5 pipelines : 4 valeurs de régularisation `C` combinées avec ou sans
`class_weight='balanced'`, soit 8 combinaisons par pipeline. Les embeddings eux-mêmes
ne sont pas ajustés — les poids pré-entraînés sont figés, seul le classifieur varie.
L'option `class_weight='balanced'` est particulièrement attendue sur les pipelines GloVe
et FastText, dont le recall classe 1 était faible en baseline.

In [ ]:
tuning_rows = []
tuning_failures = []
tuned_metrics_rows = []

for nom_pipeline, pipeline in pipelines.items():
    print("=" * 100)
    print(f"Tuning accéléré -> {nom_pipeline}")

    param_grid = classifier_param_grids.get(nom_pipeline)
    if param_grid is None:
        tuning_failures.append({"pipeline": nom_pipeline, "error": "Grille d'hyperparamètres absente"})
        print("Aucune grille trouvée.")
        continue

    try:
        preprocessor, clf_name, base_estimator = split_pipeline_preprocessor_estimator(pipeline)

        transformed = fit_transform_preprocessor_once(
            preprocessor=preprocessor,
            X_fit=X_fit,
            y_fit=y_fit,
            X_val=X_val,
        )

        best_params, tuning_results_df = tune_classifier_on_fixed_features(
            base_estimator=base_estimator,
            param_grid=param_grid,
            X_fit=transformed["X_fit_transformed"],
            y_fit=y_fit,
            X_val=transformed["X_val_transformed"],
            y_val=y_val,
            primary_metric=PRIMARY_TUNING_METRIC,
        )

        tuning_results_df.insert(0, "pipeline", nom_pipeline)
        tuning_results_df.insert(1, "classifier_name", clf_name)

        best_preprocessor_full, _, best_estimator_template = split_pipeline_preprocessor_estimator(pipeline)
        best_preprocessor_full.fit(X_train, y_train)
        X_train_vec = best_preprocessor_full.transform(X_train)
        X_test_vec = best_preprocessor_full.transform(X_test)

        best_estimator = base_estimator.set_params(**best_params)
        best_estimator.fit(X_train_vec, y_train)

        train_pred = best_estimator.predict(X_train_vec)
        test_pred = best_estimator.predict(X_test_vec)
        train_score = safe_scores(best_estimator, X_train_vec)
        test_score = safe_scores(best_estimator, X_test_vec)

        final_metrics = evaluate_refit_outputs(
            pipeline_name=nom_pipeline,
            y_train=y_train,
            y_test=y_test,
            train_pred=train_pred,
            test_pred=test_pred,
            train_score=train_score,
            test_score=test_score,
        )
        final_metrics["best_params"] = json.dumps(best_params, ensure_ascii=False)
        final_metrics["best_val_primary_score"] = float(tuning_results_df.iloc[0]["primary_score"])
        final_metrics["best_val_f1_class_1"] = float(tuning_results_df.iloc[0]["val_f1_class_1"])
        final_metrics["best_val_recall_class_1"] = float(tuning_results_df.iloc[0]["val_recall_class_1"])
        final_metrics["best_val_f1_macro"] = float(tuning_results_df.iloc[0]["val_f1_macro"])
        final_metrics["best_val_balanced_accuracy"] = float(tuning_results_df.iloc[0]["val_balanced_accuracy"])

        tuning_rows.append(tuning_results_df.iloc[0].to_dict() | {
            "pipeline": nom_pipeline,
            "best_params": json.dumps(best_params, ensure_ascii=False),
        })
        tuned_metrics_rows.append(final_metrics)

        tuning_results_path = TUNING_OUTPUT_DIR / f"{nom_pipeline}_tuning_validation_results.csv"
        tuning_results_df.to_csv(tuning_results_path, index=False)

        if USE_MLFLOW:
            log_tuning_run_to_mlflow(
                run_name=nom_pipeline,
                notebook_name="CLASSICAL",
                family_name="classical_sentence_embeddings",
                best_params=best_params,
                tuning_results_df=tuning_results_df,
                final_metrics=final_metrics,
                output_dir=TUNING_OUTPUT_DIR,
            )

        print("Meilleurs paramètres :", best_params)
        print("Meilleur score de validation :", tuning_results_df.iloc[0]["primary_score"])

    except Exception as exc:
        tuning_failures.append({"pipeline": nom_pipeline, "error": str(exc)})
        print(f"Échec tuning pour {nom_pipeline} : {exc}")


Tuning accéléré -> P16_GloVeTwitterMean_LogReg
Meilleurs paramètres : {'C': 2.0, 'class_weight': 'balanced'}
Meilleur score de validation : 0.616793893129771
Tuning accéléré -> P17_GloVeTwitterMean_LinearSVC
Meilleurs paramètres : {'C': 2.0, 'class_weight': 'balanced'}
Meilleur score de validation : 0.6348228043143297
Tuning accéléré -> P18_FastTextMean_LogReg
Meilleurs paramètres : {'C': 2.0, 'class_weight': 'balanced'}
Meilleur score de validation : 0.5945945945945946
Tuning accéléré -> P19_SentenceTransformer_LogReg


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/121 [00:00<?, ?it/s]

Batches:   0%|          | 0/22 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/143 [00:00<?, ?it/s]

Batches:   0%|          | 0/36 [00:00<?, ?it/s]

Meilleurs paramètres : {'C': 0.25, 'class_weight': 'balanced'}
Meilleur score de validation : 0.6501547987616099


Pour chaque pipeline, le préprocesseur est ajusté une seule fois sur le split d'entraînement,
puis le classifieur est optimisé sur les features déjà transformées. Le meilleur classifieur
est ensuite ré-entraîné sur l'intégralité du train vectorisé, évalué sur le test, et loggué
dans MLflow.

Les meilleurs paramètres trouvés :
- **P16 (GloVe + LogReg)** : `C=2.0, class_weight='balanced'` — score val : 0.617
- **P17 (GloVe + LinearSVC)** : `C=2.0, class_weight='balanced'` — score val : 0.635
- **P18 (FastText + LogReg)** : `C=2.0, class_weight='balanced'` — score val : 0.595
- **P19 (SentenceTransformer + LogReg)** : `C=0.5, class_weight='balanced'` — score val : 0.650
- **P20 (SentenceTransformer + LinearSVC)** : `C=0.25, class_weight='balanced'` — score val : 0.650

`class_weight='balanced'` est retenu pour les 5 pipelines sans exception — le déséquilibre
81/19 est suffisamment marqué pour que le rééquilibrage soit systématiquement bénéfique,
quelle que soit la représentation utilisée. À noter que les SentenceTransformers préfèrent
une régularisation plus forte (`C` faible) que GloVe et FastText, ce qui suggère que leurs
représentations denses de 384 dimensions sont plus susceptibles de surapprentissage avec un
classifieur peu contraint.

In [ ]:
tuning_best_df = round_results(pd.DataFrame(tuning_rows))
display(tuning_best_df)

tuned_results_df = round_results(pd.DataFrame(tuned_metrics_rows))
display(tuned_results_df)

if tuning_failures:
    print("\nPipelines tuning en échec :")
    display(pd.DataFrame(tuning_failures))


pipeline,P16_GloVeTwitterMean_LogReg,P17_GloVeTwitterMean_LinearSVC,P18_FastTextMean_LogReg,P19_SentenceTransformer_LogReg,P20_SentenceTransformer_LinearSVC
classifier_name,clf,clf,clf,clf,clf
params,"{""C"": 2.0, ""class_weight"": ""balanced""}","{""C"": 2.0, ""class_weight"": ""balanced""}","{""C"": 2.0, ""class_weight"": ""balanced""}","{""C"": 0.5, ""class_weight"": ""balanced""}","{""C"": 0.25, ""class_weight"": ""balanced""}"
primary_metric,f1_pos,f1_pos,f1_pos,f1_pos,f1_pos
primary_score,0.6168,0.6348,0.5946,0.6502,0.6502
val_accuracy,0.8161,0.8264,0.8022,0.8337,0.8344
val_precision_macro,0.7249,0.7360,0.7109,0.7449,0.7452
val_recall_macro,0.8081,0.8205,0.7935,0.8325,0.8315
val_f1_macro,0.7479,0.7605,0.7319,0.7706,0.7709
val_precision_weighted,0.8638,0.8707,0.8555,0.8772,0.8768
val_recall_weighted,0.8161,0.8264,0.8022,0.8337,0.8344


pipeline,P16_GloVeTwitterMean_LogReg,P17_GloVeTwitterMean_LinearSVC,P18_FastTextMean_LogReg,P19_SentenceTransformer_LogReg,P20_SentenceTransformer_LinearSVC
train_accuracy,0.8356,0.8406,0.8234,0.8367,0.8434
train_precision_macro,0.7447,0.7505,0.7335,0.7513,0.7584
train_recall_macro,0.8254,0.8323,0.8199,0.8486,0.8562
train_f1_macro,0.7698,0.7763,0.7578,0.7781,0.7862
train_precision_weighted,0.8741,0.8780,0.8702,0.8854,0.8897
train_recall_weighted,0.8356,0.8406,0.8234,0.8367,0.8434
train_f1_weighted,0.8471,0.8516,0.8370,0.8498,0.8557
train_precision_class_0,0.9507,0.9534,0.9511,0.9648,0.9673
train_recall_class_0,0.8417,0.8455,0.8255,0.8297,0.8359
train_f1_class_0,0.8929,0.8962,0.8839,0.8922,0.8968


Affichage des résultats de validation (meilleurs paramètres par pipeline) et des métriques
finales après ré-entraînement sur le train complet.

Sur le **set de test**, en se concentrant sur la classe 1 (Disaster) :
- **P16 (GloVe + LogReg)** et **P17 (GloVe + LinearSVC)** ressortent comme les meilleurs
  pipelines après tuning, avec les F1 classe 1 les plus élevés (0.667 et 0.665) et les
  meilleures balanced accuracy (0.848 et 0.846). Le rééquilibrage a fortement amélioré
  leur recall classe 1 (0.861 et 0.858), au prix d'une précision plus faible (~0.54).
- **P18 (FastText + LogReg)** suit avec un F1 classe 1 de 0.655 et un recall de 0.856 —
  le tuning a bien rattrapé son retard de baseline.
- **P19 et P20 (SentenceTransformer)** sont paradoxalement moins bons après tuning que
  les GloVe/FastText : F1 classe 1 de 0.640 et 0.635, avec une balanced accuracy autour
  de 0.83. Leur régularisation plus forte (`C` faible) a limité leur capacité à bien
  séparer les classes.

⚠️ Les écarts train/test sont raisonnables sur tous les pipelines, sans overfitting marqué.
Fait notable : les embeddings GloVe, entraînés sur Twitter, tirent mieux profit du
rééquilibrage que les SentenceTransformers sur ce corpus — leurs représentations semblent
mieux adaptées au registre informel des tweets disaster.

In [ ]:
comparison_df = compare_baseline_vs_tuned(
    baseline_df=pd.DataFrame(resultats),
    tuned_df=pd.DataFrame(tuned_metrics_rows),
)
display(round_results(comparison_df))


,baseline_pipeline,baseline_train_accuracy,baseline_train_precision_macro,baseline_train_recall_macro,baseline_train_f1_macro,baseline_train_precision_weighted,baseline_train_recall_weighted,baseline_train_f1_weighted,baseline_train_precision_class_0,baseline_train_recall_class_0,baseline_train_f1_class_0,baseline_train_support_class_0,baseline_train_precision_class_1,baseline_train_recall_class_1,baseline_train_f1_class_1,baseline_train_support_class_1,baseline_train_balanced_accuracy,baseline_train_roc_auc,baseline_train_pr_auc,baseline_test_accuracy,baseline_test_precision_macro,baseline_test_recall_macro,baseline_test_f1_macro,baseline_test_precision_weighted,baseline_test_recall_weighted,baseline_test_f1_weighted,baseline_test_precision_class_0,baseline_test_recall_class_0,baseline_test_f1_class_0,baseline_test_support_class_0,baseline_test_precision_class_1,baseline_test_recall_class_1,baseline_test_f1_class_1,baseline_test_support_class_1,baseline_test_balanced_accuracy,baseline_test_roc_auc,baseline_test_pr_auc,tuned_pipeline,tuned_train_accuracy,tuned_train_precision_macro,tuned_train_recall_macro,tuned_train_f1_macro,tuned_train_precision_weighted,tuned_train_recall_weighted,tuned_train_f1_weighted,tuned_train_precision_class_0,tuned_train_recall_class_0,tuned_train_f1_class_0,tuned_train_support_class_0,tuned_train_precision_class_1,tuned_train_recall_class_1,tuned_train_f1_class_1,tuned_train_support_class_1,tuned_train_balanced_accuracy,tuned_train_roc_auc,tuned_train_pr_auc,tuned_test_accuracy,tuned_test_precision_macro,tuned_test_recall_macro,tuned_test_f1_macro,tuned_test_precision_weighted,tuned_test_recall_weighted,tuned_test_f1_weighted,tuned_test_precision_class_0,tuned_test_recall_class_0,tuned_test_f1_class_0,tuned_test_support_class_0,tuned_test_precision_class_1,tuned_test_recall_class_1,tuned_test_f1_class_1,tuned_test_support_class_1,tuned_test_balanced_accuracy,tuned_test_roc_auc,tuned_test_pr_auc,tuned_best_params,tuned_best_val_primary_score,tuned_best_val_f1_class_1,tuned_best_val_recall_class_1,tuned_best_val_f1_macro,tuned_best_val_balanced_accuracy,delta_test_f1_class_1,delta_test_recall_class_1,delta_test_f1_macro,delta_test_balanced_accuracy
0,P16_GloVeTwitterMean_LogReg,0.8764,0.8474,0.7069,0.7480,0.8702,0.8764,0.8610,0.8837,0.9768,0.9279,7405.0000,0.8112,0.4370,0.5680,1691.0000,0.7069,0.8969,0.7143,0.8641,0.8108,0.6895,0.7251,0.8529,0.8641,0.8479,0.8779,0.9676,0.9206,1851.0000,0.7436,0.4113,0.5297,423.0000,0.6895,0.9103,0.6926,P16_GloVeTwitterMean_LogReg,0.8356,0.7447,0.8254,0.7698,0.8741,0.8356,0.8471,0.9507,0.8417,0.8929,7405.0000,0.5386,0.8090,0.6467,1691.0000,0.8254,0.9064,0.7257,0.8404,0.7541,0.8481,0.7811,0.8854,0.8404,0.8526,0.9633,0.8358,0.8950,1851.0000,0.5449,0.8605,0.6673,423.0000,0.8481,0.9141,0.6903,"{""C"": 2.0, ""class_weight"": ""balanced""}",0.6168,0.6168,0.7953,0.7479,0.8081,0.1376,0.4492,0.0560,0.1587
1,P17_GloVeTwitterMean_LinearSVC,0.8890,0.8504,0.7536,0.7887,0.8827,0.8890,0.8801,0.9018,0.9691,0.9343,7405.0000,0.7989,0.5381,0.6431,1691.0000,0.7536,0.9101,0.7515,0.8720,0.8141,0.7226,0.7546,0.8623,0.8720,0.8612,0.8908,0.9606,0.9244,1851.0000,0.7374,0.4846,0.5849,423.0000,0.7226,0.9124,0.7067,P17_GloVeTwitterMean_LinearSVC,0.8406,0.7505,0.8323,0.7763,0.8780,0.8406,0.8516,0.9534,0.8455,0.8962,7405.0000,0.5476,0.8190,0.6564,1691.0000,0.8323,0.9140,0.7431,0.8391,0.7526,0.8464,0.7795,0.8845,0.8391,0.8515,0.9626,0.8347,0.8941,1851.0000,0.5426,0.8582,0.6648,423.0000,0.8464,0.9114,0.6892,"{""C"": 2.0, ""class_weight"": ""balanced""}",0.6348,0.6348,0.8110,0.7605,0.8205,0.0800,0.3735,0.0248,0.1238
2,P18_FastTextMean_LogReg,0.8677,0.8352,0.6829,0.7227,0.8602,0.8677,0.8487,0.8750,0.9772,0.9233,7405.0000,0.7954,0.3885,0.5221,1691.0000,0.6829,0.8797,0.6757,0.8575,0.8014,0.6690,0.7039,0.8448,0.8575,0.8378,0.8705,0.9692,0.9172,1851.0000,0.7324,0.3688,0.4906,423.0000,0.6690,0.8878,0.6497,P18_FastTextMean_LogReg,0.8234,0.7335,0.8199,0.7578,0.8702,0.8234,0.8370,0.9511,0.8255,0.8839,7405.000

Tableau comparatif baseline vs tuned. Les gains varient selon les pipelines :

- **P18 (FastText + LogReg)** : la plus grande progression du notebook — gain de **+0.165
  sur le F1 classe 1** et **+0.487 sur le recall classe 1**. Un pipeline qui ratait plus
  de 60 % des tweets disaster en baseline devient très compétitif après tuning.
- **P16 (GloVe + LogReg)** : gain solide de **+0.138 sur le F1 classe 1** et **+0.449
  sur le recall**, avec une balanced accuracy en hausse de +0.159. Le rééquilibrage a
  bien joué son rôle.
- **P17 (GloVe + LinearSVC)** : progression notable de **+0.080 sur le F1 classe 1** et
  **+0.374 sur le recall** — déjà le meilleur pipeline GloVe en baseline, il reste solide
  après tuning.
- **P19 (SentenceTransformer + LogReg)** : gain plus modeste sur le F1 classe 1 (+0.023)
  malgré un bon recall (+0.310) — la précision chute, ce qui limite l'amélioration nette.
- **P20 (SentenceTransformer + LinearSVC)** : quasi stagnation sur le F1 classe 1 (+0.010)
  et même une légère baisse sur le F1 macro (-0.016) — ce pipeline avait déjà atteint un
  bon équilibre en baseline et le tuning n'a pas réussi à faire mieux.

En conclusion, **les embeddings GloVe et FastText sont les grands bénéficiaires du tuning**,
tandis que les SentenceTransformers, déjà bien calibrés en baseline, progressent peu.
**P16 et P17 (GloVe)** ressortent comme les pipelines les plus solides de ce notebook
après optimisation, avec le meilleur équilibre entre toutes les métriques sur le test.

In [ ]:
pd.DataFrame(tuning_rows).to_csv(TUNING_OUTPUT_DIR / f"{OUTPUT_STEM}_tuning_resume.csv", index=False)
pd.DataFrame(tuned_metrics_rows).to_csv(TUNING_OUTPUT_DIR / f"{OUTPUT_STEM}_tuned_final_results.csv", index=False)
pd.DataFrame(tuning_failures).to_csv(TUNING_OUTPUT_DIR / f"{OUTPUT_STEM}_tuning_failures.csv", index=False)

comparison_df.to_csv(TUNING_OUTPUT_DIR / f"{OUTPUT_STEM}_baseline_vs_tuned.csv", index=False)

print("Exports tuning enregistrés dans :", TUNING_OUTPUT_DIR)


Exports tuning enregistrés dans : outputs/NB4/tuning


Tous les résultats du tuning sont sauvegardés dans `outputs/NB4/tuning` : le résumé des
combinaisons testées, les métriques finales des modèles tunés, les éventuels échecs, et
le tableau comparatif baseline vs tuned. Ces fichiers serviront de référence pour la
comparaison globale entre tous les notebooks.

In [ ]:
!zip -r outputs.zip outputs

  adding: outputs/ (stored 0%)
  adding: outputs/mlruns/ (stored 0%)
  adding: outputs/mlruns/0/ (stored 0%)
  adding: outputs/mlruns/0/meta.yaml (deflated 30%)
  adding: outputs/mlruns/389536812591853163/ (stored 0%)
  adding: outputs/mlruns/389536812591853163/1e9e4bdaf2734244aa3900b4895da8df/ (stored 0%)
  adding: outputs/mlruns/389536812591853163/1e9e4bdaf2734244aa3900b4895da8df/metrics/ (stored 0%)
  adding: outputs/mlruns/389536812591853163/1e9e4bdaf2734244aa3900b4895da8df/metrics/test_pr_auc (stored 0%)
  adding: outputs/mlruns/389536812591853163/1e9e4bdaf2734244aa3900b4895da8df/metrics/test_roc_auc (stored 0%)
  adding: outputs/mlruns/389536812591853163/1e9e4bdaf2734244aa3900b4895da8df/metrics/test_f1_macro (stored 0%)
  adding: outputs/mlruns/389536812591853163/1e9e4bdaf2734244aa3900b4895da8df/metrics/train_recall_class_1 (stored 0%)
  adding: outputs/mlruns/389536812591853163/1e9e4bdaf2734244aa3900b4895da8df/metrics/train_f1_class_1 (stored 0%)
  adding: outputs/mlruns/3895368

Ce zip archive l'intégralité du dossier `outputs/` — résultats baseline, fichiers de tuning,
runs MLflow et exports CSV/XLSX — en un seul fichier téléchargeable depuis Colab. Utile à
faire avant de fermer la session pour ne rien perdre.

Si certaines dépendances comme PyTorch ou SentenceTransformer posent problème localement, les pipelines concernés peuvent échouer dans la phase baseline ou tuning, mais les autres continueront de s’exécuter et d’être exportés.